# The Agent Loop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/knivok/ai-playbook/blob/main/notebooks/00_foundations/02_agent_loop.ipynb)

---

Every agentic framework — LangGraph, CrewAI, AutoGen, all of them — is an implementation of the same underlying loop.
Understanding this loop is the single most important concept in this book.

## The ReAct Loop

ReAct (**Re**asoning + **Act**ing) is the dominant agent pattern. The agent alternates between:

```
THOUGHT → ACTION → OBSERVATION → THOUGHT → ACTION → OBSERVATION → ... → FINAL ANSWER
```

- **THOUGHT**: The LLM reasons about what to do next (often in a scratchpad)
- **ACTION**: The agent calls a tool with specific arguments
- **OBSERVATION**: The tool returns a result, which feeds back into context

This loop continues until the agent decides it has enough information to produce a final answer.

```{admonition} Why this matters for cost
:class: warning
Every iteration of the loop consumes tokens. An agent that loops 10 times uses ~10x the tokens of a single call.
Always set a `max_iterations` or `recursion_limit` in production.
```

## Setup

Install dependencies and configure your API key.

In [ ]:
# Run this cell first — installs everything needed for this chapter
!pip install -q langchain langchain-anthropic langchain-openai python-dotenv rich

In [ ]:
import os
from getpass import getpass

# Set your API key — use Anthropic Claude or OpenAI
# For Knovik engineers: retrieve from Bitwarden under 'Engineering / Shared API Keys'
if not os.getenv("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")

print("✅ API key set")

## Build the Agent Loop from Scratch

Before using any framework, let's build a minimal ReAct agent manually.
This makes the abstractions in LangGraph and CrewAI much easier to understand.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from rich.console import Console
from rich.panel import Panel
import json

console = Console()

# ── Define tools ──────────────────────────────────────────────────────────────

@tool
def search_web(query: str) -> str:
    """Search the web for information about a topic. Returns a summary."""
    # Simulated results for demo purposes
    results = {
        "smart home market size": "The global smart home market was valued at $121B in 2024, projected to reach $338B by 2030. Key growth drivers: energy efficiency, security, and convenience.",
        "apple homekit competitors": "Apple HomeKit competes with Google Home, Amazon Alexa, Samsung SmartThings, and Home Assistant. HomeKit differentiates on privacy and Apple ecosystem integration.",
        "zigbee hub demand": "Zigbee remains dominant in smart home protocols with 60%+ device compatibility. Matter protocol is rising but Zigbee hubs still required for legacy devices.",
    }
    for key in results:
        if key in query.lower():
            return results[key]
    return f"Search results for '{query}': Market is growing, competition is moderate, opportunity exists for differentiated products."

@tool
def calculate_market_score(market_size_billions: float, competition_level: str, growth_rate_percent: float) -> dict:
    """Calculate a niche viability score based on market metrics."""
    competition_multiplier = {"low": 1.0, "medium": 0.7, "high": 0.4}.get(competition_level.lower(), 0.5)
    score = min(100, (market_size_billions * 0.3 + growth_rate_percent * 2) * competition_multiplier)
    return {
        "viability_score": round(score, 1),
        "recommendation": "PURSUE" if score > 50 else "RESEARCH MORE" if score > 30 else "AVOID",
        "reasoning": f"Score driven by ${market_size_billions}B market, {competition_level} competition, {growth_rate_percent}% growth"
    }

tools = [search_web, calculate_market_score]
tools_by_name = {t.name: t for t in tools}

print("✅ Tools defined:", [t.name for t in tools])

In [ ]:
# ── Build the minimal agent loop ──────────────────────────────────────────────

llm = ChatAnthropic(model="claude-3-5-sonnet-20241022", temperature=0)
llm_with_tools = llm.bind_tools(tools)

def run_agent(goal: str, max_iterations: int = 5) -> str:
    """Run a minimal ReAct agent loop."""
    messages = [HumanMessage(content=goal)]
    iteration = 0

    console.print(Panel(f"[bold cyan]GOAL:[/] {goal}", title="Agent Starting"))

    while iteration < max_iterations:
        iteration += 1
        console.print(f"\n[dim]── Iteration {iteration} ──[/dim]")

        # THOUGHT + ACTION: LLM decides what to do
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        # Check if the agent is done (no tool calls = final answer)
        if not response.tool_calls:
            console.print(Panel(
                f"[bold green]{response.content}[/]",
                title="✅ Final Answer",
                border_style="green"
            ))
            return response.content

        # OBSERVATION: Execute each tool call and feed results back
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]

            console.print(f"  [yellow]ACTION:[/] {tool_name}({json.dumps(tool_args, indent=2)})")

            result = tools_by_name[tool_name].invoke(tool_args)

            console.print(f"  [blue]OBSERVATION:[/] {result}")

            messages.append(ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"]
            ))

    return "Max iterations reached without a final answer."

# ── Run it ────────────────────────────────────────────────────────────────────
result = run_agent(
    "Research the smart home / Zigbee hub niche and give me a viability score and recommendation."
)

## What Just Happened?

Walk through the output above and you will see the ReAct loop in action:

1. The LLM received the goal and decided to **call `search_web`** to get market data
2. The tool returned results, which became the **observation**
3. The LLM used that observation to **call `calculate_market_score`** with specific numbers
4. The final observation gave the agent enough information to **produce a final answer**

This is exactly what LangGraph, CrewAI, and every other framework in this book is doing under the hood — they just add structure, state management, multi-agent coordination, and production-grade error handling on top.

---

## Try it yourself

Modify the `goal` string and rerun the cell. Try:
- `"Research the Apple HomeKit accessory market. Should Knovik expand into this niche?"`
- `"Is the Matter protocol hub market worth entering in 2026?"`

Notice how the agent picks different tools and different search queries depending on the goal.

In [ ]:
# ✏️ Your turn — change the goal and rerun
result = run_agent(
    "YOUR GOAL HERE"
)

---

## Key Takeaways

- The ReAct loop (Thought → Action → Observation) is the foundation of all agentic frameworks
- An agent stops looping when the LLM produces a response with no tool calls
- `max_iterations` is not optional in production — always set it
- Every framework in this book is a structured, opinionated wrapper around this same pattern

Next: {doc}`03_memory_and_tools` — how agents remember things across turns and what good tool design looks like.